In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
file_path = "last_6_months.csv"
df = pd.read_csv(file_path)

In [3]:
df.columns = ["data_in", "data_exec", "description", "value", "total"]
df.head()

,data_in,data_exec,description,value,total
0,4/24/2025,4/24/2025,RENDIMENTOS DE CLIENTES RBRX11 S/ 98,"R$ 7,84","R$ 523,58"
1,4/17/2025,4/17/2025,RENDIMENTOS DE CLIENTES VGIR11 S/ 85,"R$ 10,20","R$ 515,74"
2,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES URPR11 S/ 11,"R$ 8,80","R$ 505,54"
3,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES MXRF11 S/ 68,"R$ 6,12","R$ 496,74"
4,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES KIVO11 S/ 10,"R$ 9,00","R$ 490,62"


In [4]:
df["name"] = df["description"].str.extract(r"\b([A-Z]{4,5}\d{2})\b")
df["qnt"] = df["description"].str.extract(r"\s(\d+)$").astype(float)
df

,data_in,data_exec,description,value,total,name,qnt
0,4/24/2025,4/24/2025,RENDIMENTOS DE CLIENTES RBRX11 S/ 98,"R$ 7,84","R$ 523,58",RBRX11,98.0
1,4/17/2025,4/17/2025,RENDIMENTOS DE CLIENTES VGIR11 S/ 85,"R$ 10,20","R$ 515,74",VGIR11,85.0
2,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES URPR11 S/ 11,"R$ 8,80","R$ 505,54",URPR11,11.0
3,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES MXRF11 S/ 68,"R$ 6,12","R$ 496,74",MXRF11,68.0
4,4/14/2025,4/14/2025,RENDIMENTOS DE CLIENTES KIVO11 S/ 10,"R$ 9,00","R$ 490,62",KIVO11,10.0
...,...,...,...,...,...,...,...
74,10/11/2024,10/11/2024,RENDIMENTOS DE CLIENTES HABT11 S/ 9,"R$ 9,00","R$ 300,03",HABT11,9.0
75,10/8/2024,10/8/2024,RENDIMENTOS DE CLIENTES CACR11 S/ 12,"R$ 15,84","R$ 291,03",CACR11,12.0
76,10/8/2024,10/8/2024,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,"R$ 17,52","R$ 275,19",AIEC11,24.0
77,10/7/2024,10/7/2024,RENDIMENTOS DE CLIENTES LIFE11 S/ 101,"R$ 13,63","R$ 257,67",LIFE11,101.0


In [5]:
df["value"] = (
    df["value"]
    .str.replace("R$", "", regex=False)
    .str.replace("-",  "", regex=False)
    .str.replace(".",  "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

df["qnt"] = df["qnt"].astype(float)

df["value/unit"] = df["value"] / df["qnt"]

In [6]:
ativos_unicos = df["name"].dropna().unique()
ativos_unicos

array(['RBRX11', 'VGIR11', 'URPR11', 'MXRF11', 'KIVO11', 'KNCR11',
       'HABT11', 'CACR11', 'AIEC11', 'LIFE11', 'ARRI11'], dtype=object)

In [7]:
precos_fiis = {}

for ativo in ativos_unicos:
    ticker = yf.Ticker(f"{ativo}.SA")  # .SA = B3
    dados = ticker.history(period="1d")
    if not dados.empty:
        preco_fechamento = dados["Close"].iloc[-1]
        precos_fiis[ativo] = preco_fechamento
    else:
        precos_fiis[ativo] = None  # caso não encontre


In [10]:
df["price"] = df["name"].map(precos_fiis)
df['ratio'] = df["value/unit"] / df["price"] * 100
df["data_in"] = pd.to_datetime(df["data_in"], format="%m/%d/%Y")
df["data_exec"] = pd.to_datetime(df["data_exec"], format="%m/%d/%Y")
df["total_value"] = df['price'].astype(float) * df['qnt'].astype(float)
df

,data_in,data_exec,description,value,total,name,qnt,value/unit,price,ratio,total_value
0,2025-04-24,2025-04-24,RENDIMENTOS DE CLIENTES RBRX11 S/ 98,7.84,"R$ 523,58",RBRX11,98.0,0.08000,8.130000,0.984010,796.740011
1,2025-04-17,2025-04-17,RENDIMENTOS DE CLIENTES VGIR11 S/ 85,10.20,"R$ 515,74",VGIR11,85.0,0.12000,9.460000,1.268499,804.100003
2,2025-04-14,2025-04-14,RENDIMENTOS DE CLIENTES URPR11 S/ 11,8.80,"R$ 505,54",URPR11,11.0,0.80000,51.980000,1.539053,571.779995
3,2025-04-14,2025-04-14,RENDIMENTOS DE CLIENTES MXRF11 S/ 68,6.12,"R$ 496,74",MXRF11,68.0,0.09000,9.270000,0.970874,630.360031
4,2025-04-14,2025-04-14,RENDIMENTOS DE CLIENTES KIVO11 S/ 10,9.00,"R$ 490,62",KIVO11,10.0,0.90000,67.470001,1.333926,674.700012
...,...,...,...,...,...,...,...,...,...,...,...
74,2024-10-11,2024-10-11,RENDIMENTOS DE CLIENTES HABT11 S/ 9,9.00,"R$ 300,03",HABT11,9.0,1.00000,81.500000,1.226994,733.500000
75,2024-10-08,2024-10-08,RENDIMENTOS DE CLIENTES CACR11 S/ 12,15.84,"R$ 291,03",CACR11,12.0,1.32000,96.769997,1.364059,1161.239960
76,2024-10-08,2024-10-08,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,17.52,"R$ 275,19",AIEC11,24.0,0.73000,44.750000,1.631285,1074.000000
77,2024-10-07,2024-10-07,RENDIMENTOS DE CLIENTES LIFE11 S/ 101,13.63,"R$ 257,67",LIFE11,101.0,0.13495,8.710000,1.549374,879.710004


In [11]:
df_abril = df[df["data_in"].dt.month == 4].sort_values(by="ratio")

df_abril


,data_in,data_exec,description,value,total,name,qnt,value/unit,price,ratio,total_value
8,2025-04-08,2025-04-08,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,7.68,"R$ 448,34",AIEC11,24.0,0.32,44.750000,0.715084,1074.000000
3,2025-04-14,2025-04-14,RENDIMENTOS DE CLIENTES MXRF11 S/ 68,6.12,"R$ 496,74",MXRF11,68.0,0.09,9.270000,0.970874,630.360031
0,2025-04-24,2025-04-24,RENDIMENTOS DE CLIENTES RBRX11 S/ 98,7.84,"R$ 523,58",RBRX11,98.0,0.08,8.130000,0.984010,796.740011
5,2025-04-11,2025-04-11,RENDIMENTOS DE CLIENTES KNCR11 S/ 8,8.32,"R$ 481,62",KNCR11,8.0,1.04,103.050003,1.009219,824.400024
10,2025-04-07,2025-04-07,RENDIMENTOS DE CLIENTES ARRI11 S/ 88,7.92,"R$ 428,54",ARRI11,88.0,0.09,7.420000,1.212938,652.960007
6,2025-04-11,2025-04-11,RENDIMENTOS DE CLIENTES HABT11 S/ 9,9.00,"R$ 473,30",HABT11,9.0,1.00,81.500000,1.226994,733.500000
1,2025-04-17,2025-04-17,RENDIMENTOS DE CLIENTES VGIR11 S/ 85,10.20,"R$ 515,74",VGIR11,85.0,0.12,9.460000,1.268499,804.100003
4,2025-04-14,2025-04-14,RENDIMENTOS DE CLIENTES KIVO11 S/ 10,9.00,"R$ 490,62",KIVO11,10.0,0.90,67.470001,1.333926,674.700012
7,2025-04-08,2025-04-08,RENDIMENTOS DE CLIENTES CACR11 S/ 12,15.96,"R$ 464,30",CACR11,12.0,1.33,96.769997,1.374393,1161.239960
9,2025-04-07,2025-04-07,RENDIMENTOS DE CLIENTES LIFE11 S/ 101,12.12,"R$ 440,66",LIFE11,101.0,0.12,8.710000,1.377727,879.710004


In [14]:
soma = df_abril['value'].sum()
total = df_abril["total_value"].sum()

soma / total * 100

1.1695361662817654

In [ ]:
lista_de_dfs = [
    {name: group.reset_index(drop=True)}
    for name, group in df.groupby("name")
]

In [ ]:
# Acessar o DataFrame correspondente ao primeiro name
primeiro = lista_de_dfs[0]
nome = list(primeiro.keys())[0]
df_filtrado = primeiro[nome]

# print(f"Nome: {nome}")
df_filtrado

,data_in,data_exec,description,value,total,name,qnt,value/unit,price,ratio
0,4/8/2025,4/8/2025,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,7.68,"R$ 448,34",AIEC11,24.0,0.32,45.07,0.710007
1,3/12/2025,3/12/2025,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,7.68,"R$ 345,29",AIEC11,24.0,0.32,45.07,0.710007
2,2/10/2025,2/10/2025,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,4.08,"R$ 242,29",AIEC11,24.0,0.17,45.07,0.377191
3,1/9/2025,1/9/2025,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,29.28,"R$ 141,51",AIEC11,24.0,1.22,45.07,2.706900
4,12/9/2024,12/9/2024,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,14.88,"R$ 14,88",AIEC11,24.0,0.62,45.07,1.375638
5,11/8/2024,11/8/2024,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,14.88,"R$ 1.479,60",AIEC11,24.0,0.62,45.07,1.375638
6,10/8/2024,10/8/2024,RENDIMENTOS DE CLIENTES AIEC11 S/ 24,17.52,"R$ 275,19",AIEC11,24.0,0.73,45.07,1.619703
